# Plot BT Task Trees (All Tasks)

Plots all parsed behavior trees side-by-side per task.
- Iterates through all `task_*.json` files in one model `tasks/` folder
- Prints `task_id` before plotting (easy to Ctrl+F)
- Skips trees that fail parsing
- Skips tasks with zero parsed trees
- Uses widened node spacing so labels are visible

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt

In [ ]:
# Ensure repo root is importable even if notebook cwd is elsewhere
start = Path.cwd()
repo_root = None
for p in [start, *start.parents]:
    if (p / "src").exists() and (p / "results").exists():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not locate repo root containing src/ and results/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.parsers.bt import parse_bt_output

In [ ]:
# Same results layout as Plot_BT_Tasks_Maps.ipynb
results_root = repo_root / "results"
model_rel = Path("wall_bt_2026-04-29_15-21-54/Qwen/Qwen2.5-3B-Instruct")
model_dir = model_rel if model_rel.is_absolute() else (results_root / model_rel)
tasks_dir = model_dir / "tasks"

if not tasks_dir.exists():
    raise FileNotFoundError(f"Missing tasks dir: {tasks_dir}")

task_files = sorted(tasks_dir.glob("task_*.json"))
if not task_files:
    raise ValueError(f"No task_*.json files found in: {tasks_dir}")

print(f"Model dir: {model_dir}")
print(f"Total task files: {len(task_files)}")

In [ ]:
def node_children(node: dict) -> list:
    if node.get("type") in ("sequence", "fallback"):
        return node.get("children", [])
    return []


def node_label(node: dict) -> str:
    t = node.get("type", "?")
    if t == "condition":
        obs = node.get("observation", "?")
        exp = node.get("expected", True)
        return f"cond\n{obs} == {exp}"
    if t == "action":
        call = node.get("call", {})
        tool = call.get("tool_name", "?")
        args = call.get("arguments", {})
        return f"action\n{tool}({args})"
    if t in ("sequence", "fallback"):
        return t
    return str(t)


def tree_stats(root: dict) -> tuple[int, int]:
    """Return (leaf_count, max_depth) for adaptive spacing."""
    def walk(node: dict, depth: int) -> tuple[int, int]:
        children = node_children(node)
        if not children:
            return 1, depth
        leaf_total = 0
        depth_max = depth
        for c in children:
            l, d = walk(c, depth + 1)
            leaf_total += l
            depth_max = max(depth_max, d)
        return leaf_total, depth_max

    return walk(root, 0)


def layout_tree(root: dict, x_spacing: float, y_spacing: float):
    positions = {}
    edges = []
    leaf_x = 0

    def walk(node: dict, depth: int, parent_id=None):
        nonlocal leaf_x
        node_id = id(node)
        children = node_children(node)

        if not children:
            x = leaf_x * x_spacing
            leaf_x += 1
        else:
            child_xs = []
            for c in children:
                cx, _ = walk(c, depth + 1, node_id)
                child_xs.append(cx)
            x = sum(child_xs) / len(child_xs)

        y = -depth * y_spacing
        positions[node_id] = (x, y, node)
        if parent_id is not None:
            edges.append((parent_id, node_id))
        return x, y

    walk(root, depth=0)
    return positions, edges


def draw_tree(ax, root: dict, title: str):
    leaves, depth = tree_stats(root)

    # Adaptive spacing for larger trees
    x_spacing = 2.6 if leaves <= 8 else 3.2
    y_spacing = 2.2 if depth <= 5 else 2.8

    pos, edges = layout_tree(root, x_spacing=x_spacing, y_spacing=y_spacing)

    for p, c in edges:
        x1, y1, _ = pos[p]
        x2, y2, _ = pos[c]
        ax.plot([x1, x2], [y1, y2], color="gray", linewidth=1)

    for _, (x, y, node) in pos.items():
        ntype = node.get("type")
        if ntype == "condition":
            fc = "#dbeafe"
        elif ntype == "action":
            fc = "#dcfce7"
        elif ntype == "sequence":
            fc = "#fde68a"
        elif ntype == "fallback":
            fc = "#fbcfe8"
        else:
            fc = "#e5e7eb"

        ax.text(
            x,
            y,
            node_label(node),
            ha="center",
            va="center",
            fontsize=7,
            bbox={"boxstyle": "round,pad=0.4", "facecolor": fc, "edgecolor": "black"},
        )

    ax.set_title(title)
    ax.axis("off")

    xs = [v[0] for v in pos.values()]
    ys = [v[1] for v in pos.values()]
    if xs and ys:
        ax.set_xlim(min(xs) - 3.0, max(xs) + 3.0)
        ax.set_ylim(min(ys) - 3.0, max(ys) + 2.0)

In [ ]:
def plot_trees_for_task_type(task_type: str, start_version: int = 1, end_version: int = 20):
    plotted_tasks = 0
    skipped_tasks = 0

    for version in range(start_version, end_version + 1):
        task_id = f"{task_type}_v{version}"
        task_file = tasks_dir / f"task_{task_id}.json"
        if not task_file.exists():
            continue

        task_data = json.loads(task_file.read_text(encoding="utf-8"))
        behavior_trees = task_data.get("behavior_trees", [])

        parsed_roots = []
        for bt in behavior_trees:
            idx = bt.get("bt_index")
            raw = bt.get("llm_output", "")
            plan, err = parse_bt_output(raw)
            if err is not None:
                continue
            parsed_roots.append((idx, plan.root.model_dump()))

        if not parsed_roots:
            skipped_tasks += 1
            continue

        # Searchable task marker in output
        print(f"\n=== task_id: {task_id} ===")

        n = len(parsed_roots)

        # Scale figure size with tree complexity to reduce clutter
        max_leaves = max(tree_stats(root)[0] for _, root in parsed_roots)
        max_depth = max(tree_stats(root)[1] for _, root in parsed_roots)
        per_tree_width = max(7.5, min(14.0, 4.0 + 0.35 * max_leaves))
        fig_height = max(8.0, min(15.0, 5.0 + 0.7 * max_depth))

        fig, axes = plt.subplots(1, n, figsize=(per_tree_width * n, fig_height), squeeze=False)
        axes = axes[0]
        for ax, (idx, root) in zip(axes, parsed_roots):
            draw_tree(ax, root, title=f"tree_{idx}")

        fig.suptitle(f"Task: {task_id} | Parsed trees: {n}")
        plt.tight_layout()
        plt.show()
        plotted_tasks += 1

    print(f"\n{task_type}: plotted={plotted_tasks}, skipped_no_parsed={skipped_tasks}")

## go_to_target (v1-v20)

In [ ]:
plot_trees_for_task_type("go_to_target", 1, 20)

## face_target (v1-v20)

In [ ]:
plot_trees_for_task_type("face_target", 1, 20)

## move_to_closest_target (v1-v20)

In [ ]:
plot_trees_for_task_type("move_to_closest_target", 1, 20)

## go_to_multiple_targets (v1-v20)

In [ ]:
plot_trees_for_task_type("go_to_multiple_targets", 1, 20)

## go_around_obstacle (v1-v20)

In [ ]:
plot_trees_for_task_type("go_around_obstacle", 1, 20)